In [ ]:
from bcnexus import constants as bcnexus_const
from bcnexus.clews import model_structure

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# local packages

from bcnexus import utils

- Configure the scenario name and timeslices from the available results

In [ ]:
storage_algorithm="Kotzur"
nexus_scenario='Base_CNZ_noCCS'
timeslices=8
solver="gurobi"

In [ ]:
from bcnexus.clews.datapackage import GetDataPackage
nexus_results_root=Path(f'../results/clews/Model_{storage_algorithm}_{nexus_scenario}/{timeslices}ts_csvs_{solver}')
result_pack=GetDataPackage(nexus_results_root)
result_pack.show

---

TEMP

In [ ]:
# Land use

region=model_structure.Regions['REGION1'][0]

crops = {}
# DEFINE THAT THE NAMING CONVENTION FILE NEED TO LIST CROPS ALPHABETICAL

water_supply = model_structure.IrrigationTypeList

input_level = model_structure.IntensityList
mode_crop_combo=bcnexus_const.mode_of_operation

## Land use sector

In [ ]:
TotalAnnualTechnologyActivityByMode_df=result_pack.get_dataframe('TotalAnnualTechnologyActivityByMode')
Landuse_df=TotalAnnualTechnologyActivityByMode_df[TotalAnnualTechnologyActivityByMode_df['TECHNOLOGY'].str.startswith('LNDAGR')].drop('REGION', axis=1)
Landuse_df['MODE_OF_OPERATION'] = Landuse_df['MODE_OF_OPERATION'].astype(int)
Landuse_df['crop_combo'] = Landuse_df['MODE_OF_OPERATION'].map(mode_crop_combo)
Landuse_df['land_use'] = Landuse_df['crop_combo']#.str[0:4]
Landuse_df.drop(['MODE_OF_OPERATION','crop_combo'], axis=1, inplace=True)

In [ ]:
# Exclude rows where land_use appears among the dict values
crops_total_df = Landuse_df[~Landuse_df['land_use'].isin(model_structure.LandUseCodes.values())]

In [ ]:
# crops_total_df = crops_total_df[crops_total_df['land_use'].str.startswith('CP')]
crops_total_df_pivot = crops_total_df.pivot_table(index='YEAR', 
                                            columns='land_use',
                                            values='VALUE', 
                                            aggfunc='sum').reset_index().fillna(0)
# crops_total_df_pivot = crops_total_df_pivot.reindex(sorted(crops_total_df.columns), axis=1).set_index('YEAR')

In [ ]:
import pandas as pd
import plotly.express as px

def df_plot(df, y_title, p_title):
    # Melt wide columns into long form for stacked plotting
    df_melted = df.melt(
        id_vars=['YEAR'],
        var_name='Crop',
        value_name='Area'
    )

    fig = px.bar(
        df_melted,
        x='YEAR',
        y='Area',
        color='Crop',
        barmode='stack',
        labels={'Area': y_title, 'YEAR': 'Year'},
        title=p_title
    )
    fig.update_layout(
        legend_title_text='Crop',
        xaxis=dict(type='category'),
        yaxis=dict(title=y_title)
    )
    fig.show()


In [ ]:
crop_cols = [c for c in crops_total_df_pivot.columns if c != 'YEAR']

# Map each crop column to its 3-letter prefix
prefix_map = {col: col[:3] for col in crop_cols}

# Group columns with same prefix and sum their values
df_grouped = crops_total_df_pivot.groupby(prefix_map,axis=1).sum()
df_grouped['YEAR'] = crops_total_df_pivot['YEAR']

crops_total_df_pivot=df_grouped

In [ ]:
df_plot(crops_total_df_pivot,'Land area (1000 sq.km.)','Area by crop')

In [ ]:
# Exclude rows where land_use appears among the dict values
land_df = Landuse_df[Landuse_df['land_use'].isin(model_structure.LandUseCodes.values())]

In [ ]:
land_df = land_df.pivot_table(index='YEAR', 
                                          columns='land_use',
                                          values='VALUE', 
                                          aggfunc='sum').reset_index().fillna(0)
# land_total_df['AGR'] = 0


In [ ]:
land_df['AGR'] = 0
for crop in crop_cols:
    if crop in land_df.columns:
        land_df['AGR'] += land_df[crop]
        land_df.drop(crop, axis=1, inplace=True)
# land_df = land_df.reindex(sorted(land_df.columns), axis=1).set_index('y').reset_index().rename(columns=det_col).astype('float64')

if not land_df.empty:
    df_plot(land_df,'Land area (1000 sq.km.)','Area by land cover type')

In [ ]:
ProductionByTechnologyAnnual_df=result_pack.get_dataframe('ProductionByTechnologyAnnual')

In [ ]:
crops_prod_df = ProductionByTechnologyAnnual_df[ProductionByTechnologyAnnual_df['FUEL'].str.startswith('CRP')]
# crops_prod_df.loc[:, 'FUEL'] = crops_prod_df['FUEL'].map(bcnexus_const.name_mapping['FUEL'])
crops_prod_df.loc[:, 'FUEL'] = crops_prod_df['FUEL'].str[3:6]

In [ ]:
crops_prod_df_pivot = crops_prod_df.pivot_table(index='YEAR', 
                                          columns='FUEL',
                                          values='VALUE',
                                          aggfunc='sum').reset_index().fillna(0)
crops_prod_df_pivot = crops_prod_df_pivot.reindex(sorted(crops_prod_df_pivot.columns), axis=1).set_index('YEAR').reset_index()
# crops_prod_df['YEAR'] = years

if len(crops_prod_df_pivot.columns) > 1:
    df_plot(crops_prod_df_pivot,'Production (Million tonnes)','Crop production')

In [ ]:
crops_yield_df = crops_prod_df_pivot/crops_total_df_pivot

In [ ]:
crops_yield_df['YEAR']=crops_prod_df_pivot['YEAR']

In [ ]:
import plotly.express as px

# Multiply all numeric columns except 'YEAR' by 10
crops_yield_df.loc[:, crops_yield_df.columns != 'YEAR'] = (
    crops_yield_df.loc[:, crops_yield_df.columns != 'YEAR'] * 10
)

# Plot only if multiple columns exist
if len(crops_yield_df.columns) > 1:
    # Melt the dataframe to long format
    df_melted = crops_yield_df.melt(
        id_vars='YEAR',
        var_name='Crop',
        value_name='Yield'
    )

    # Create interactive line+marker plot
    fig = px.line(
        df_melted,
        x='YEAR',
        y='Yield',
        color='Crop',
        markers=True,
        labels={'YEAR': 'Year', 'Yield': 'Yield (t/ha)'},
        title='Yield (tonnes per hectare)'
    )

    fig.update_traces(marker=dict(size=8))
    fig.update_layout(
        xaxis=dict(type='category'),
        yaxis=dict(title='Yield (t/ha)')
    )
    fig.show()


----

In [ ]:
nexus_ts_plots={}
plot_categories = ['Climate', 'Land', 'Energy', 'Water', str(timeslices)]
nexus_plots = {category: {} for category in plot_categories}
nexus_plots[f'{timeslices}'] = nexus_ts_plots
plots = {'nexus': nexus_plots}


In [ ]:
from bcnexus.vis import plot_Climate

nexus_climate_plots={}
nexus_plots['Climate'] = nexus_climate_plots

nexus_climate_plots['emission_total']=plot_Climate.get_total_annual_emission(result_pack.get_dataframe('AnnualEmissions'), nexus_scenario)
nexus_climate_plots['emission_by_source']=plot_Climate.get_emission_from_fuels(result_pack.get_dataframe('AnnualTechnologyEmission'), nexus_scenario)
nexus_climate_plots['emission_by_sector']=plot_Climate.get_emission_from_sector(result_pack.get_dataframe('AnnualTechnologyEmission'), nexus_scenario)

In [ ]:
from bcnexus.vis import plot_Land
nexus_plots['Land'] = plot_Land.plot_landuse_for_clusters(result_pack.get_dataframe('RateOfProductionByTechnologyByMode'), nexus_scenario)

In [ ]:
from bcnexus.vis import plot_Energy
nexus_energy_plots={}
nexus_plots['Energy'] = nexus_energy_plots

nexus_energy_plots["Nexus_sectoral_consumption"] , nexus_energy_plots["Nexus_fuel_consumption"] = plot_Energy.plot_combined_stacked_energy_consumption(result_pack.get_dataframe('UseByTechnology'), 
                                                                                                                                                       'gwh',
                                                                                                                                                       nexus_scenario)
nexus_energy_plots["Nexus_generation_from_fuels"]=plot_Energy.get_annual_generation_plot(result_pack.get_dataframe('ProductionByTechnology'), 
                                                                                  nexus_scenario)
nexus_energy_plots["Nexus_capacity_investments"]=plot_Energy.get_capacity_plot(result_pack.get_dataframe('NewCapacity'),
                                                                               nexus_scenario,
                                                                               investment=True)
nexus_energy_plots["Nexus_capacity_total"]=plot_Energy.get_capacity_plot(result_pack.get_dataframe('TotalCapacityAnnual'),nexus_scenario,investment=False)
nexus_energy_plots["Nexus_power_generation_timeslices"]=plot_Energy.get_generation_timeseries_plot(result_pack.get_dataframe('RateOfUseByTechnology'),24,nexus_scenario)
nexus_energy_plots["Nexus_power_generation_annual"]=plot_Energy.get_annual_power_generation_plot(result_pack.get_dataframe('ProductionByTechnologyAnnual'),nexus_scenario)
nexus_energy_plots["Nexus_capital_investment_power"]=plot_Energy.get_capital_investments(result_pack.get_dataframe('CapitalInvestment'),nexus_scenario)

In [ ]:
nexus_energy_plots["Nexus_power_generation_annual"]

In [ ]:
df = utils.add_power_tech_labels(result_pack.get_dataframe('CapitalInvestment'),'capacity')
df.power_techs_labels.unique()

In [ ]:
nexus_energy_plots["Nexus_capital_investment_power"]

In [ ]:
result_pack.get_dataframe('CapitalInvestment').TECHNOLOGY.unique()

In [ ]:
data=result_pack.get_dataframe('CapitalInvestment')

In [ ]:
data_pwr = data.loc[data['TECHNOLOGY'].str.startswith('PWR')].copy()
data_pwr.loc[:, 'TECHNOLOGY_CODE'] = data_pwr['TECHNOLOGY'].str[:6]
data_pwr.loc[:, 'TECHNOLOGY_LABEL'] = data_pwr['TECHNOLOGY_CODE'].map(bcnexus_const.legend_labels)
data_pwr
data_pwr_grouped_data = data_pwr.groupby(['TECHNOLOGY_LABEL','TECHNOLOGY_CODE', 'YEAR', 'REGION']).sum().reset_index()


In [ ]:
data_pwr_grouped_data

In [ ]:

import plotly.express as px

# Create a bar plot with custom legend colors
# custom_colors = bcnexus_const.custom_colors
fig = px.bar(data_pwr_grouped_data, x='YEAR', y='VALUE', color='TECHNOLOGY_LABEL', 
             labels={'VALUE': 'Mill $'}, title='Bar Plot of VALUE over Years')



fig.show()